# KCS 前20日半导体出口同比：Stage1 现金相对 SOXX 五日增量信息门

原39项与共同月份/来源年龄控制，唯一新增KCS原报同比；保守Qlib时钟下的条件性开发检验。
首次执行仅因父帧终端交易日超出来源日历而在拟合前停止；修复账见attempt1_failure.json与repair_plan.json。


In [ ]:
import hashlib,json,os,pathlib,shlex
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from src.analysis.seed_disagreement_information import full_calendar_bootstrap
from src.data.nas_market_data import archive_nas_recorder,_settings,_ssh
root=pathlib.Path.cwd();card=root/'research/stage1_kcs_early20_cash_h5_information_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest();assert plan['frozen'] and plan['execution_authorized_by_active_Goal'] and not (card/'run_state.json').exists() and json.loads((card/'repair_plan.json').read_text())['status']=='approved_mechanical_terminal_unknown_mask_only'
parent=root/src['parent_card'];kcard=root/src['KCS_source_map_card'];p_art=parent/'qlib_recorder'/src['parent_experiment_id']/src['parent_recorder_id']/'artifacts';k_art=kcard/'qlib_recorder'/src['KCS_source_map_experiment_id']/src['KCS_source_map_recorder_id']/'artifacts'
for p,h in [(parent/'summary.json',src['parent_summary_sha256']),(p_art/'inputs/features',src['parent_features_sha256']),(p_art/'inputs/daily_frame',src['parent_frame_sha256']),(p_art/'inputs/timing',src['parent_timing_sha256']),(p_art/'models/base_outer_71',src['parent_model_sha256']),(kcard/'summary.json',src['KCS_source_map_summary_sha256']),(k_art/'analysis/mapped_calendar',src['KCS_mapped_calendar_artifact_sha256']),(root/'src/analysis/seed_disagreement_information.py',src['bootstrap_module_sha256'])]:assert sha(p)==h,(p,sha(p),h)
assert json.loads((kcard/'summary.json').read_text())['NAS_scoped_restore']=='nas_scoped_restore.json' and not json.loads((kcard/'summary.json').read_text())['limitations'][0].startswith('PIT qualified')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1);assert len(D.calendar(start_time='2011-01-03',end_time='2026-10-09',freq='day'))==3966
with R.uri_context(str(parent/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=src['parent_experiment_id'],recorder_id=src['parent_recorder_id']);assert rec.status=='FINISHED';features=rec.load_object('inputs/features');frame=rec.load_object('inputs/daily_frame');timing=rec.load_object('inputs/timing');model=rec.load_object('models/base_outer_71')
with R.uri_context(str(kcard/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=src['KCS_source_map_experiment_id'],recorder_id=src['KCS_source_map_recorder_id']);assert rec.status=='FINISHED';mapped=rec.load_object('analysis/mapped_calendar')
assert frame.index.equals(features.index) and frame.index.equals(timing.index) and len(frame)==3966 and len(mapped)==3966;base=list(model.base_features);assert len(base)==39 and set(base).issubset(features.columns)
trades=pd.DatetimeIndex(pd.to_datetime(timing.decision_at).dt.tz_localize(None).dt.normalize());source=mapped.set_index('trade_date').reindex(trades).reset_index(drop=True);source.index=frame.index;assert int(source.source_known.isna().sum())==1 and source.index[source.source_known.isna()][0]==pd.Timestamp('2026-10-09');source['source_known']=source.source_known.fillna(False).astype(bool);assert ((source.loc[source.source_known,'portal_registered_date'].to_numpy(dtype='datetime64[ns]') < trades[source.source_known.to_numpy(bool)].to_numpy(dtype='datetime64[ns]')).all())
parts=[frame.soxx_forward_open_return.astype(float).shift(-i) for i in range(5)];valid=np.logical_and.reduce([np.isfinite(x.to_numpy(float))&(x.to_numpy(float)>-1) for x in parts]);target=(-np.log1p(pd.concat(parts,axis=1)).sum(axis=1)).where(valid);maturity=pd.to_datetime(timing.return_available_at.shift(-4),utc=True);coh=plan['cohorts'];train_calendar=frame.index.to_series().between(*coh['train_signal_span']).to_numpy();held_calendar=frame.index.to_series().between(*coh['held_signal_span']).to_numpy();train_cut=pd.Timestamp('2023-01-01',tz='America/New_York').tz_convert('UTC');held_cut=pd.Timestamp('2026-10-10',tz='America/New_York').tz_convert('UTC')
train_mask=train_calendar&source.source_known.to_numpy(bool)&maturity.lt(train_cut).to_numpy()&target.notna().to_numpy();held_mask=held_calendar&source.source_known.to_numpy(bool)&maturity.lt(held_cut).to_numpy()&target.notna().to_numpy();assert int(train_mask.sum())==coh['train_common_mature_source_known']==1240 and int(held_mask.sum())==coh['held_common_mature_source_known']==940 and int(held_calendar.sum())==coh['held_full_calendar']==944
extra='kcs_semiconductor_export_yoy_pct';month=trades.month.to_numpy(float);f=features[base].copy();f['common_calendar_sin']=np.sin(2*np.pi*(month-1)/12);f['common_calendar_cos']=np.cos(2*np.pi*(month-1)/12);f['common_KCS_source_age_sessions']=source.source_age_sessions.to_numpy(float);f[extra]=source.kcs_semiconductor_export_yoy_pct.to_numpy(float);baseline=base+['common_calendar_sin','common_calendar_cos','common_KCS_source_age_sessions'];meta=pd.DataFrame({'train_mature_source_known':train_mask,'held_mature_source_known':held_mask,'source_known':source.source_known.to_numpy(bool),'source_age_sessions':source.source_age_sessions.to_numpy(float),'source_event':source.source_event_eligible_today.to_numpy(bool),'label_available_at':maturity},index=frame.index);static=pd.concat({'feature':f,'label':pd.DataFrame({'cash_minus_SOXX_H5':target},index=frame.index),'meta':meta},axis=1);static.index=pd.MultiIndex.from_arrays([frame.index,['SOXL_SOXX_CASH']*len(frame)],names=['datetime','instrument']);dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':tuple(coh['train_signal_span']),'held':tuple(coh['held_signal_span'])})
tr=dataset.prepare('train',col_set=['feature','label','meta']);he=dataset.prepare('held',col_set=['feature','label','meta']);train=tr.loc[tr[('meta','train_mature_source_known')].astype(bool)];held=he.loc[he[('meta','held_mature_source_known')].astype(bool)];assert len(train)==1240 and len(he)==944 and len(held)==940 and np.isfinite(train[('label','cash_minus_SOXX_H5')]).all() and np.isfinite(held[('label','cash_minus_SOXX_H5')]).all()
maxcorr=float(features.loc[frame.index[train_mask],base].corrwith(source.loc[train_mask,extra],method='spearman').abs().max()) if extra in source else float(features.loc[frame.index[train_mask],base].corrwith(source.loc[train_mask,'kcs_semiconductor_export_yoy_pct'],method='spearman').abs().max());assert maxcorr<0.8
preflight={'status':'pinned_Stage1_39_plus_common_controls_KCS_cash_H5_cohorts_matched','plan_sha256':sha(card/'plan.json'),'train_mature':len(train),'held_calendar':len(he),'held_scored':len(held),'held_unknown_label_tail':4,'train_KCS_event_sessions':int(source.loc[train_calendar,'source_event_eligible_today'].sum()),'held_KCS_event_sessions':int(source.loc[held_calendar,'source_event_eligible_today'].sum()),'max_abs_train_original39_Spearman':maxcorr,'historical_source_byte_vintage_qualified':False,'model_fits':0,'native_accounts':0};assert preflight['train_KCS_event_sessions']==60 and preflight['held_KCS_event_sessions']==45;(card/'preflight_result.json').write_text(json.dumps(preflight,indent=2)+'\n')
state={'status':'fit_started','plan_sha256':preflight['plan_sha256'],'fit_attempts':0,'fit_completed':0,'bootstrap_attempts':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
xtr0=train['feature'][baseline].to_numpy(float);xhe0=held['feature'][baseline].to_numpy(float);xtr1=train['feature'][baseline+[extra]].to_numpy(float);xhe1=held['feature'][baseline+[extra]].to_numpy(float);y=train[('label','cash_minus_SOXX_H5')].to_numpy(float);yh=held[('label','cash_minus_SOXX_H5')].to_numpy(float);assert np.isfinite(xtr1[:,-1]).all() and np.isfinite(xhe1[:,-1]).all()
models=[]
for x in (xtr0,xtr1):
 m=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=1.0));state['fit_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');m.fit(x,y);models.append(m);state['fit_completed']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
control,candidate=models;p0=control.predict(xhe0);p1=candidate.predict(xhe1);mean=float(y.mean());coef=float(candidate.named_steps['ridge'].coef_[-1]);dates=pd.DatetimeIndex(held.index.get_level_values('datetime'));scored=pd.DataFrame({'signal_date':dates,'label':yh,'baseline_prediction':p0,'candidate_prediction':p1,'train_mean_prediction':mean});scored['gain_vs_baseline']=(yh-p0)**2-(yh-p1)**2;scored['gain_vs_train_mean']=(yh-mean)**2-(yh-p1)**2;full=pd.DataFrame({'signal_date':pd.DatetimeIndex(he.index.get_level_values('datetime'))}).merge(scored,on='signal_date',how='left',validate='one_to_one');assert len(full)==944 and full.label.notna().sum()==940
metrics={'candidate_MSE':float(np.square(yh-p1).mean()),'baseline_MSE':float(np.square(yh-p0).mean()),'train_mean_MSE':float(np.square(yh-mean).mean()),'added_standardized_coefficient':coef,'comparators':{}};gates={'coefficient_negative':bool(coef<0)};reps={};indices=None;boot=plan['information_gate']['bootstrap']
for key in ('baseline','train_mean'):
 diff=scored['gain_vs_'+key];halves=[]
 for pos in (slice(0,472),slice(472,944)):
  group=full.iloc[pos];halves.append({'first_date':str(group.signal_date.iloc[0].date()),'last_date':str(group.signal_date.iloc[-1].date()),'calendar_rows':len(group),'scored_rows':int(group.label.notna().sum()),'mean_gain':float(group['gain_vs_'+key].mean())})
 annual=[]
 for year in (2024,2025):
  group=scored.loc[scored.signal_date.dt.year.eq(year)];annual.append({'year':year,'rows':len(group),'mean_gain':float(group['gain_vs_'+key].mean())})
 state['bootstrap_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');interval,idx,rep=full_calendar_bootstrap(full['gain_vs_'+key].to_numpy(float),mean_block=boot['mean_block_sessions'],resamples=boot['draws_each'],seed=boot['seed'],alpha=boot['one_sided_alpha_each']);assert interval['valid_replicates']==boot['draws_each']
 if indices is None:indices=idx
 else:assert np.array_equal(indices,idx)
 reps[key]=rep;metrics['comparators'][key]={'full_mean_gain':float(diff.mean()),'halves':halves,'complete_years':annual,'bootstrap':interval};gates[key]={'full_positive':bool(diff.mean()>0),'both_halves_positive':bool(all(z['mean_gain']>0 for z in halves)),'both_2024_2025_positive':bool(all(z['mean_gain']>0 for z in annual)),'corrected_lower_positive':bool(interval['lower_bound']>0)}
passed=bool(gates['coefficient_negative'] and all(all(v.values()) for v in gates.values() if isinstance(v,dict)))
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();dataset.config(dump_all=True,recursive=True);rec.save_objects(artifact_path='inputs',plan=plan,preflight=preflight,dataset=dataset);rec.save_objects(artifact_path='models',baseline=control,candidate=candidate);rec.save_objects(artifact_path='predictions',scored=scored);rec.save_objects(artifact_path='analysis',full_calendar=full,metrics=metrics,gates=gates,bootstrap_replicates=reps,bootstrap_indices=indices,information_gate_pass=passed);rec.log_metrics(fits=2,native_accounts=0,information_gate_pass=int(passed),gain_vs_baseline=metrics['comparators']['baseline']['full_mean_gain']);rid,eid=rec.id,str(rec.experiment_id)
state['status']='finished';(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');summary={'experiment':plan['experiment'],'decision':'conditional_KCS_early20_cash_H5_information_pass_needs_historical_vintage_and_0bp_account' if passed else 'not_promoted_fixed_KCS_early20_cash_H5_information_failed','objective_achieved':False,'plan_sha256':preflight['plan_sha256'],'preflight':preflight,'metrics':metrics,'gates':gates,'information_gate_pass':passed,'new_model_fits':2,'new_bootstraps':2,'new_native_accounts':0,'historical_first_public_byte_vintage_qualified':False,'historical_independence':False,'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','models/*','predictions/*','analysis/*']},'limitations':['KCS historical original first-public byte versions are unqualified; source map only uses conservative portal-date clock.','Reported raw YoY is working-day/base sensitive, and same-day source carry repeats 60/45 monthly events across daily rows.','H5 information MSE is not a closed-loop Stage1 SOXX account or independent forward evidence.']};(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);remote=_ssh(_settings(root)['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0];assert remote==evidence['sha256'];summary['NAS_evidence']=evidence;summary['NAS_readback']={'sha256':remote,'verified':True};(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n');print(json.dumps({'decision':summary['decision'],'preflight':preflight,'metrics':metrics,'gates':gates,'recorder_id':rid,'NAS_sha256':remote},ensure_ascii=False,indent=2))


## 中文结论

在原Stage1 **39项输入**及共同月份相位/来源年龄控制上，仅新增已按官方登记日之后首个Qlib美股会话映射的韩国海关半导体出口原报同比。同一批成熟且来源已知的训练日 **1,240**、重复使用的留出日 **940**，完整留出日历944日；对应原报训练60月、留出45月。训练原39项与该来源最高绝对秩相关约 **0.252**，未因相关性挑信号或改公式。该来源是未经工作日调整的当前可下载版本，历史首次公开字节仍未认证。

新增标准化系数 **+0.007033**，与冻结负号假设相反。候选留出MSE **0.003597**，原39项加共同控制 **0.002876**，训练均值 **0.002353**；对两控制的完整配对MSE改善分别 **-0.000720214／-0.001244166**，校正区块下界 **-0.002701946／-0.003701588**。2024和2025两个完整年度对原39对照的均值差分别 **-0.000251554／-0.000453426**，均未通过；2026部分年恶化更大，但失败不依赖单独剔除该年。所有四年和15季度的误差归因在 [failure_attribution.json](failure_attribution.json)，不产生择时开关。

首次执行在拟合前因父数据终端信号2026-10-09对应交易日10-12超出已钉住的来源日历而停止，消耗父Recorder读取4、KCS来源读取1、Qlib日历1；[机械修复](repair_plan.json)只将该**评价区间外**终端日标未知。续跑后累计读取为父8、KCS2、日历2；正式 **2次拟合、2次10,000抽样区块检验、0资金账户**。原生Qlib Recorder `79c39de340004cada3cf59a51d636f44` 的模型/预测/门工件已从NAS SHA-256 `cca3ee092325fde9806f0271d303ce3e29326fe66a62eeb46c244915cf6635b6` 限定冷恢复一致。

决定 **`not_promoted_fixed_KCS_early20_cash_H5_information_failed`**。不翻同比符号、改工作日口径、选年份、去极值或调模型/风险阈值救援此固定版本。信息门失败，不运行完整Stage1 0bp Tiered候选账户；因此 SOXX净财富、回撤、年度、126日窗口、2/5bp与独立前向硬门仍未由本路线满足。Goal继续寻找实质不同机制。
